# Clasificación de Deserción Estudiantil con SVM sobre Dataset Sintético (11,500 Estudiantes)

**Proyecto**: Machine Learning II - Universidad 2026-II  
**Autor**: Arnold Santiago Torres  
**Dataset**: `estudiantes_sinteticos_11500.csv` (11,500 registros, 21 variables socioeconómicas y académicas)  

---  
### Contenido del Notebook:
1. **Análisis Exploratorio de Datos (EDA) Previo**: Estadísticas, distribuciones de la deserción estudiantil y top correlaciones.
2. **Evaluación de Normalización vs Baseline (Sin Normalizar)**: Comparación cuantitativa de 5 métodos de normalización vs el modelo **Baseline sin escalar** con ejecución paralela multihilo (`n_jobs=2`).
3. **Importancia de Variables (SVM)**: Identificación de las variables más influyentes tomadas por SVM (Permutation Importance con `n_jobs=2`).
4. **Visualización del Hiperplano de SVM y Vectores de Soporte**: Frontera de decisión no lineal ($w^T \phi(x) + b = 0$), márgenes y vectores de soporte en 2D (PCA y variables clave).
5. **Clasificación de Estudiantes**: Matriz de confusión, métricas globales (Accuracy, Precision, Recall, F1, ROC-AUC) y distribución de distancias funcionales al hiperplano.
---

In [ ]:
import os
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from joblib import Parallel, delayed

from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    f1_score, precision_score, recall_score, accuracy_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, MaxAbsScaler,
    QuantileTransformer, PowerTransformer, OrdinalEncoder, OneHotEncoder
)
from scipy.stats import ttest_ind

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['font.sans-serif'] = 'Arial'

N_JOBS = 2  # Usar 2 núcleos del procesador
print(f"Configuración de ejecuciones utilizando {N_JOBS} núcleos de la CPU.")

## 1. Análisis Exploratorio de Datos (EDA) Previo

In [ ]:
data_path = 'estudiantes_sinteticos_11500.csv'
df = pd.read_csv(data_path)

# Formular variable objetivo binaria 'desercion'
riesgo = (
    (df['promedio_academico'] < 3.0).astype(int) * 2.5 +
    (df['materias_reprobadas'] >= 2).astype(int) * 2.5 +
    (df['puntaje_icfes_percentil'] < 40).astype(int) * 1.0 +
    (df['trabaja_mientras_estudia'] == 1).astype(int) * 1.0 +
    (df['beneficiario_beca'] == 0).astype(int) * 0.5 +
    (df['estrato_socioeconomico'] <= 2).astype(int) * 0.5
)
df['desercion'] = (riesgo >= 3.5).astype(int)

cat_cols = ['sexo', 'estado_civil', 'zona_residencia', 'departamento',
            'nivel_educativo_padre', 'nivel_educativo_madre', 'sector_ies',
            'nivel_formacion', 'metodologia', 'area_conocimiento']
num_cols = ['edad_ingreso', 'promedio_academico', 'materias_reprobadas',
            'distancia_hogar_ies_km', 'puntaje_icfes_percentil', 'semestre_cursado',
            'estrato_socioeconomico', 'anio_registro', 'trabaja_mientras_estudia',
            'beneficiario_icetex', 'beneficiario_beca']

ohe = OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')
X_cat_encoded = ohe.fit_transform(df[cat_cols])
cat_feature_names = ohe.get_feature_names_out(cat_cols)

X_num = df[num_cols].values
X_raw = pd.DataFrame(np.hstack([X_num, X_cat_encoded]), columns=num_cols + list(cat_feature_names))
y = df['desercion'].values

print(f"Dimensiones: {df.shape[0]} estudiantes, {df.shape[1]} variables orginales ({X_raw.shape[1]} encodificadas).")
print(f"Valores ausentes (nulos): {df.isnull().sum().sum()}")
print("\nDistribución de la Variable Objetivo 'desercion' (1 = Desertor, 0 = No Desertor):")
print(pd.Series(y).value_counts(normalize=True) * 100)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.countplot(data=df, x='desercion', palette='Set2', ax=axes[0])
axes[0].set_xticklabels(['No Desertor (0)', 'Desertor (1)'])
axes[0].set_title('Distribución de Estudiantes (11,500 Sintéticos)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Estado Académico', fontsize=12)
axes[0].set_ylabel('Cantidad de Estudiantes', fontsize=12)

correlations = X_raw.apply(lambda col: col.corr(pd.Series(y))).sort_values()
top_corr = pd.concat([correlations.head(5), correlations.tail(5)])
colors = ['#d95f02' if c > 0 else '#7570b3' for c in top_corr.values]

axes[1].barh(top_corr.index, top_corr.values, color=colors)
axes[1].set_title('Top 10 Variables Correlacionadas con Deserción', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Coeficiente de Correlación de Pearson', fontsize=12)
axes[1].axvline(0, color='black', linestyle='--', linewidth=0.8)

plt.tight_layout()
plt.show()

## 2. Evaluación de Métodos de Normalización vs Baseline (Sin Normalizar) (`n_jobs=2`)

In [ ]:
normalization_methods = {
    'Standard Scaler': StandardScaler(),
    'Min Max Scaler': MinMaxScaler(),
    'Quantile Transformer': QuantileTransformer(n_quantiles=50, random_state=42, output_distribution='uniform'),
    'Max Absolute Scaler': MaxAbsScaler(),
    'Power Transformer': PowerTransformer(method='yeo-johnson'),
    'No Normalization (Baseline)': None
}

N_TRIALS = 10

def evaluate_single_trial(trial_idx):
    X_train, X_test, y_train, y_test = train_test_split(
        X_raw, y, test_size=0.2, random_state=42 + trial_idx, stratify=y
    )
    trial_scores = {}
    for method_name, scaler in normalization_methods.items():
        try:
            if scaler is None:
                X_tr, X_te = X_train.values, X_test.values
            else:
                scaler_inst = type(scaler)(**scaler.get_params())
                X_tr = scaler_inst.fit_transform(X_train)
                X_te = scaler_inst.transform(X_test)
            clf = SVC(kernel='rbf', C=1.0, cache_size=1000, max_iter=3000, random_state=42)
            clf.fit(X_tr, y_train)
            preds = clf.predict(X_te)
            trial_scores[method_name] = f1_score(y_test, preds)
        except Exception:
            trial_scores[method_name] = np.nan
    return trial_scores

print(f"Ejecutando {N_TRIALS} iteraciones de SVM con n_jobs={N_JOBS}...")
parallel_results = Parallel(n_jobs=N_JOBS, prefer="threads")(delayed(evaluate_single_trial)(t) for t in range(N_TRIALS))

results = {method: [] for method in normalization_methods.keys()}
for res in parallel_results:
    for method_name, score in res.items():
        results[method_name].append(score)

baseline_scores = results['No Normalization (Baseline)']
summary_rows = []
for method_name, scores in results.items():
    mean_f1 = np.mean(scores)
    std_f1 = np.std(scores)
    p_val = ttest_ind(scores, baseline_scores, equal_var=False)[1] if method_name != 'No Normalization (Baseline)' else 1.0
    summary_rows.append({
        'Normalization Method': method_name,
        'Average F1 Score': mean_f1,
        'Std Dev': std_f1,
        'P-value vs Baseline': p_val
    })

summary_df = pd.DataFrame(summary_rows).sort_values(by='Average F1 Score', ascending=False)
display(summary_df)

In [ ]:
plt.figure(figsize=(12, 6))
colors_bar = ['#d95f02' if 'Baseline' in name else '#1b9e77' for name in summary_df['Normalization Method']]
barplot = sns.barplot(data=summary_df, x='Average F1 Score', y='Normalization Method', palette=colors_bar)
plt.title('Desempeño SVM en Dataset Sintético: Normalización vs Baseline (Sin Normalizar)', fontsize=14, fontweight='bold')
plt.xlabel('F1-Score Promedio (10 iteraciones)', fontsize=12)
plt.xlim(0.4, 1.0)
plt.show()

## 3. Importancia de Variables en SVM (Permutation Importance con `n_jobs=2`)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_raw, y, test_size=0.2, random_state=42, stratify=y)
scaler_best = StandardScaler()
X_train_scaled = scaler_best.fit_transform(X_train)
X_test_scaled = scaler_best.transform(X_test)

clf_svm = SVC(kernel='rbf', C=1.0, probability=True, cache_size=1000, random_state=42)
clf_svm.fit(X_train_scaled, y_train)

perm_importance = permutation_importance(
    clf_svm, X_test_scaled, y_test, n_repeats=5, random_state=42, n_jobs=N_JOBS
)

sorted_idx = perm_importance.importances_mean.argsort()[::-1]
imp_df = pd.DataFrame({
    'Feature': X_raw.columns[sorted_idx],
    'Importance_Mean': perm_importance.importances_mean[sorted_idx],
    'Importance_Std': perm_importance.importances_std[sorted_idx]
})

plt.figure(figsize=(10, 7))
top_15_imp = imp_df.head(15)
plt.barh(top_15_imp['Feature'][::-1], top_15_imp['Importance_Mean'][::-1], color='#2b5c8f', xerr=top_15_imp['Importance_Std'][::-1])
plt.title('Top 15 Variables Más Relevantes en la Clasificación de SVM', fontsize=14, fontweight='bold')
plt.xlabel('Disminución Promedio en F1-Score al Permutar Variable', fontsize=12)
plt.tight_layout()
plt.show()
display(imp_df.head(10))

## 4. Visualización del Hiperplano de Separación y Vectores de Soporte (SVM)

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

svm_pca = SVC(kernel='rbf', C=1.0, random_state=42)
svm_pca.fit(X_train_pca, y_train)

x_min, x_max = X_train_pca[:, 0].min() - 0.5, X_train_pca[:, 0].max() + 0.5
y_min, y_max = X_train_pca[:, 1].min() - 0.5, X_train_pca[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 400), np.linspace(y_min, y_max, 400))
Z = svm_pca.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

plt.figure(figsize=(11, 8))
plt.contourf(xx, yy, Z, levels=[-np.inf, 0, np.inf], colors=['#e0f3f8', '#fee0d2'], alpha=0.6)
contours = plt.contour(xx, yy, Z, levels=[-1.0, 0.0, 1.0], linestyles=['--', '-', '--'],
                       colors=['#3182bd', '#e6550d', '#3182bd'], linewidths=[1.5, 2.5, 1.5])
plt.clabel(contours, inline=True, fontsize=10, fmt={-1.0: 'Margen (-1)', 0.0: 'Hiperplano (0)', 1.0: 'Margen (+1)'})

sample_idx = np.random.choice(len(y_test), size=1000, replace=False)
X_sample_pca = X_test_pca[sample_idx]
y_sample = y_test[sample_idx]

plt.scatter(X_sample_pca[y_sample == 0, 0], X_sample_pca[y_sample == 0, 1], c='#2b83ba', label='No Desertor (0)', alpha=0.6, edgecolors='k', s=30)
plt.scatter(X_sample_pca[y_sample == 1, 0], X_sample_pca[y_sample == 1, 1], c='#d7191c', label='Desertor (1)', alpha=0.6, edgecolors='k', s=30)

sv = svm_pca.support_vectors_
plt.scatter(sv[:, 0], sv[:, 1], s=100, facecolors='none', edgecolors='black', linewidths=1.2, label=f'Vectores de Soporte ({len(sv)} puntos)')

plt.title('Hiperplano de Separación SVM (Kernel RBF en PCA 2D - Dataset Sintético)', fontsize=14, fontweight='bold')
plt.xlabel(f'Componente Principal 1 ({pca.explained_variance_ratio_[0]*100:.1f}% Varianza)', fontsize=12)
plt.ylabel(f'Componente Principal 2 ({pca.explained_variance_ratio_[1]*100:.1f}% Varianza)', fontsize=12)
plt.legend(loc='upper right', frameon=True)
plt.tight_layout()
plt.show()

## 5. Análisis de Clasificación de Estudiantes

In [ ]:
y_pred = clf_svm.predict(X_test_scaled)
cm = confusion_matrix(y_test, y_pred)

print("REPORTE DE CLASIFICACIÓN (2,300 ESTUDIANTES DE TEST):")
print(classification_report(y_test, y_pred, target_names=['No Desertor (0)', 'Desertor (1)']))

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[0],
            xticklabels=['No Desertor (Pred)', 'Desertor (Pred)'],
            yticklabels=['No Desertor (Real)', 'Desertor (Real)'],
            annot_kws={'size': 14, 'weight': 'bold'})
axes[0].set_title('Matriz de Confusión de Estudiantes', fontsize=14, fontweight='bold')

fpr, tpr, _ = roc_curve(y_test, clf_svm.predict_proba(X_test_scaled)[:, 1])
auc_val = roc_auc_score(y_test, clf_svm.predict_proba(X_test_scaled)[:, 1])
axes[1].plot(fpr, tpr, color='#e6550d', lw=2.5, label=f'SVM RBF (AUC = {auc_val:.4f})')
axes[1].plot([0, 1], [0, 1], color='navy', lw=1.5, linestyle='--')
axes[1].set_title('Curva ROC - Discriminación', fontsize=14, fontweight='bold')
axes[1].legend(loc='lower right')

plt.tight_layout()
plt.show()

In [ ]:
decision_dist = clf_svm.decision_function(X_test_scaled)

plt.figure(figsize=(10, 6))
sns.histplot(decision_dist[y_test == 0], color='#2b83ba', kde=True, label='Estudiantes No Desertores', stat='density', alpha=0.5)
sns.histplot(decision_dist[y_test == 1], color='#d7191c', kde=True, label='Estudiantes Desertores', stat='density', alpha=0.5)

plt.axvline(0, color='black', linestyle='--', linewidth=2, label='Hiperplano f(x) = 0')
plt.axvline(1, color='gray', linestyle=':', linewidth=1.5, label='Margen +1')
plt.axvline(-1, color='gray', linestyle=':', linewidth=1.5, label='Margen -1')

plt.title('Distribución de Distancias Funcionales de Estudiantes al Hiperplano SVM', fontsize=14, fontweight='bold')
plt.xlabel('Distancia Funcional f(x) = wᵀφ(x) + b', fontsize=12)
plt.ylabel('Densidad de Estudiantes', fontsize=12)
plt.legend(loc='upper right')
plt.tight_layout()
plt.show()